# Results

In [1]:
import duckdb
import pandas as pd
import sys
import spacy
import os
sys.path.append('..')

from src.preprocess import make_corpus_preprocess, make_clean_data
from src.semantic import build_semantic_index, semantic_search
from src.bm25 import bm25_search, build_bm25

In [ ]:
# load clean data
raw_data_path = '../data/raw/merged.parquet'
clean_data_path = "../data/processed/clean_data.csv"

if not os.path.exists(clean_data_path):
    make_clean_data(raw_data_path, clean_data_path)

clean_data = pd.read_csv(clean_data_path)

## Build Corpus and Preprocessing

In [ ]:
# if corpus is already processed and saved, pass to save time
corpus_path = "../data/processed/preprocessed_corpus.csv"
if not os.path.exists(corpus_path):
    make_corpus_preprocess(clean_data_path, corpus_path)

corpus = pd.read_csv(corpus_path)

## Save Indices for BM25 and Embeddings

In [4]:
# BM25 index

pickle_path = "../data/processed/bm25.pkl"
bm25 = build_bm25(pickle_path, corpus)

In [5]:
# Semantic index 
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
semantic_index_path = '../data/processed/embedding.faiss'

if not os.path.exists(semantic_index_path):
    build_semantic_index(corpus, model, semantic_index_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieve Results

In [6]:
# Set max width for results dataframe
pd.set_option('display.max_colwidth', 200)

In [7]:
queries = ["Wet wipes",
           "Bar Soap",
           "Small hair dryer",
           "The best air humidifer with essential oil",
           "mineral sunscreen for babies",
           "hair spray that last more than 6 hours",
           "Best Vitamins or supplements to take for pregnant women",
           "equipments for stretching at home",
           "sunrise lamp that will help me to wake up in the morning",
           "Something to relieve my back pain"]

In [8]:
for q in queries:
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    display(bm25_search(q, pickle_path, corpus))

    print("\nSemantic search top results:")
    display(semantic_search(q, semantic_index_path, model, corpus))

QUERY: Wet wipes

BM25 top results:


,product_title,review_text,rating,score
4426,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (200 Wipes),that the wipes are thick and not thin.,5.0,15.452317
3174,Lens Wipes Pre-moistened Eye Glasses Cleaner Wipes 120 Individually Packaged for Cleaning Glasses Sunglasses Computer Screens Touchscreens Monitors,"I bought these based on the reviews, but they are not wet enough. My optician said that the wetter the better when it comes to cleansing your glasses. In fact, he recommended a spray and cloth bet...",1.0,12.032342
6058,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)",These were really wipes more for use in medical not full sheets for glasses.,1.0,11.516642
3256,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)","What we liked most was that it does an excellent job of cleaning your eyeglasses, better than anything we have used, BUT it is a two part system. First you use the wet one, then the dry one. The ...",2.0,11.441593
7325,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)",Do not buy this item. The wipes are so small they won't cover the lens. It's a pain that you need 2 wipes to try to clean your glasses. Save your money and buy some other product.,1.0,10.531983



Semantic search top results:


,product_title,review_text,rating,score
8276,"Pampers Baby Fresh Water Baby Wipes 3X Pop-Top Packs, 192 Count","These wipes are good for just about anything including your bum...lol. The scent is refreshing and not overpowering either. Plus, you can carry these anywhere for extra cleanliness. I highly recom...",5.0,0.940183
3626,"Travelon Hand Soap Toiletry Sheets, 50-Count",This did clean my clothes on a recent trip to Europe but then my sister used it and her hands we wet so it turned into a lump we had to just use. The direction very clearly state you must use wit...,5.0,0.938806
2185,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry Wipes 300 pcs, for Lens Eyeglasses Glasses Screen iPhone Cell Phone, Remove smudges and Dirt Effectively, no More Scratches Streaks Residue (300)",One towel to clean and one to dry. What a pain.,1.0,0.917276
2290,"Pampers Baby Fresh Water Baby Wipes 3X Pop-Top Packs, 192 Count",Decent quality but I'll stick with huggies wipes.,4.0,0.897646
4426,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (200 Wipes),that the wipes are thick and not thin.,5.0,0.705773


QUERY: Bar Soap

BM25 top results:


...

,product_title,review_text,rating,score
1996,Naturebright L6060 Per2 Led Daylight Lamp,A+ However a bit difficult to program.,4.0,1.079733
1697,"Verilux Original Natural Spectrum Deluxe Floor Lamp, Ivory","I bought this as a gift for my mom, it was at the very least a bit of a pain to put together and cheaply made. It required electrical tape and a good bit of cursing! Is now working ok. The gril...",4.0,1.073756
537,"C&A Scientific Alcohol Lamp, Metal, Wickless, 100ml (97-5320)",The upside is that this is a very well made little lamp.<br />it's easy to start and produces a hot and steady flame.<br />If you need a light duty flame hot enough to heat a test tube or steriliz...,4.0,1.060951
1646,Naturebright L6060 Per2 Led Daylight Lamp,"This functions well as daylight alarm. It's best feature is that the LED's arc out over the bed and shine down, as opposed to other models which shine straight out horizontally, which means you ha...",3.0,1.048516
8831,Naturebright L6060 Per2 Led Daylight Lamp,"I tried the Phillips previously (see review there) but just couldn't get it to work. The NatureBright is easy to figure out, the controls make a lot more sense and you can easily see that it's se...",4.0,0.732633


QUERY: Something to relieve my back pain

BM25 top results:


,product_title,review_text,rating,score
491,"Neck Stretcher Cervical Neck Traction Device Over Door for Home Use,Portable Neck Traction for Neck Pain Relief, Physical Therapy AIDS for Neck Spine Decompressor (Black)",First time user. It took me a while a find out how to assemble them. After some adjustments I find a way to use it. And it does helps relieve my neck pain.,5.0,10.952530
5899,Shoulder Wrap Gel Ice Hot Cold Pack for Shoulder Injury Pain Relief Therapy Rotator Cuff Rheumatoid Arthritis Treatment Osteoarthritis Bursitis Tendinitis AC Joint Sports Injuries,"I use it alot, great product, helps relieve my muscle pain.",5.0,10.870432
6671,"ZSZBACE Posture Corrector Back Brace for Men and Women- Relieve Back Pain, Align Spain, Correct Kyphosis (XXL)",It help with my back pain,5.0,10.828765
671,Korean Red Ginseng Patch Powerstrip Energy Pain Relief - 20 Patches,These patches really do relieve the pain. They can be cut to whatever size you need.They stick extremely well.,5.0,10.687405
3484,"Real Time Pain Relief George Foreman's Knockout Formula, 1 Oz GoPak",This stuff does exactly what it says it will do. Relieve pain and make you feel better. Happy!,5.0,10.650396



Semantic search top results:


,product_title,review_text,rating,score
1449,"Lower Back Stretcher Spine Board-Back Stretcher for Lower Back Pain Relief, Back Cracking Device, Sciatica, Scoliosis, Spine Deck, on Bed,on Chair, Yoga Mat & Car-Spine Stretcher",This really gives a nice stretch. I use it on my office chair daily. Ever since I started using this the lower back pain I would have at the end of the day is gone.,5.0,0.939688
8077,"CareU(™) Coccyx Orthopedic Portable Comfort Memory Foam Seat Cushion with Non Slip Cover- Helps Relieve Kyphotic, Hip and Sciatica Pain (Standard Size 17.7x15 Inch)",I'm sitting on it right now. love it,5.0,0.923567
1478,"Acupuncture Mat and Pillow Set-Relieve Your Stress, Back, Neck, and Sciatic Pain(99% Cotton Fabric, Plastic Spikes and Foam core) Green",Really helped with tension in my shoulders!,5.0,0.918509
7561,"Back Massager, Aptoco Back Stretcher for Sciatica Relief, Herniated Disc/Spinal Stenosis-Back Massage Stretcher, Back Stretching Device",A little uncomfortable for me (not very flexable) but my wife loves it and uses it multiple times a day.,4.0,0.910006
5550,"Genericb Back Massage Stretcher Arch Magic Message Stretcher Back Stretcher Lumbar Support Device, Lower and Upper Back Pain Relief Relax Mate Spine Pain Relief Chiropractic",A little to stiff for sore back muscles. need to use a soft pad with it,4.0,0.852649
